# 04 - 临床药师 LoRA 微调

结构与 03 完全相同，只换数据集和角色 prompt。直接顺序运行。

## Step 0: 修复 Windows 编码 + 镜像设置（必须最先跑）

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

import pathlib
_orig_read_text = pathlib.Path.read_text
def _utf8_read_text(self, encoding=None, errors=None, **kwargs):
    if encoding is None:
        encoding = 'utf-8'
    return _orig_read_text(self, encoding=encoding, errors=errors, **kwargs)
pathlib.Path.read_text = _utf8_read_text

import builtins
_orig_open = builtins.open
def _utf8_open(file, mode='r', buffering=-1, encoding=None, errors=None, **kwargs):
    if 'b' not in mode and encoding is None:
        encoding = 'utf-8'
    return _orig_open(file, mode, buffering, encoding, errors, **kwargs)
builtins.open = _utf8_open

print('环境变量 + 编码 monkey-patch 完成')

In [ ]:
import torch, gc, json, random, inspect
from pathlib import Path
from datasets import Dataset

# 清空显存（避免接着之前 notebook 的残余）
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f'CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    print(f'清空后显存占用: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## Step 1: 配置（药师专属）

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

NOTEBOOK_DIR = Path.cwd()
EVAL_DIR = NOTEBOOK_DIR.parent
BACKEND_DIR = EVAL_DIR.parent
DATA_PATH = EVAL_DIR / 'datasets' / 'data' / 'pharmacist_train.jsonl'
OUTPUT_DIR = EVAL_DIR / 'models' / 'pharmacist_lora'
ROLE_PROMPT_PATH = BACKEND_DIR / 'workspace' / 'roles' / 'PHARMACIST.md'

# 训练超参（跟 03 保持一致以便对比）
MAX_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4
NUM_EPOCHS = 3
WARMUP_RATIO = 0.05
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'v_proj', 'k_proj', 'o_proj']
USE_4BIT = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'数据: {DATA_PATH}')
print(f'输出: {OUTPUT_DIR}')
print(f'数据存在: {DATA_PATH.exists()}')
print(f'角色 prompt 存在: {ROLE_PROMPT_PATH.exists()}')

## Step 2: 加载数据

In [ ]:
random.seed(42)
records = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            r = json.loads(line)
            msgs = r.get('messages', [])
            if len(msgs) >= 3 and msgs[-1].get('role') == 'assistant':
                records.append({'messages': msgs})

random.shuffle(records)
n_val = max(20, int(len(records) * 0.05))
train_dataset = Dataset.from_list(records[n_val:])
val_dataset = Dataset.from_list(records[:n_val])
print(f'训练: {len(train_dataset)}, 验证: {len(val_dataset)}')

## Step 3: 加载基座模型 + LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 4: 训练

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_init_params = set(inspect.signature(SFTConfig.__init__).parameters.keys())
config_kwargs = dict(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=10,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to='none',
    seed=42,
)
if 'max_length' in sft_init_params:
    config_kwargs['max_length'] = MAX_LENGTH
elif 'max_seq_length' in sft_init_params:
    config_kwargs['max_seq_length'] = MAX_LENGTH
if 'packing' in sft_init_params:
    config_kwargs['packing'] = False

sft_config = SFTConfig(**config_kwargs)

trainer_init_params = set(inspect.signature(SFTTrainer.__init__).parameters.keys())
trainer_kwargs = dict(
    model=model, args=sft_config,
    train_dataset=train_dataset, eval_dataset=val_dataset,
)
if 'processing_class' in trainer_init_params:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_init_params:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)
print('开始 LoRA 微调...')
trainer.train()

## Step 5: 保存

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(OUTPUT_DIR / 'training_info.json', 'w', encoding='utf-8') as f:
    json.dump({
        'base_model': MODEL_NAME,
        'role': 'pharmacist',
        'train_size': len(train_dataset),
        'val_size': len(val_dataset),
        'epochs': NUM_EPOCHS,
        'max_length': MAX_LENGTH,
    }, f, ensure_ascii=False, indent=2)

print(f'保存到: {OUTPUT_DIR.resolve()}')
for p in OUTPUT_DIR.glob('*'):
    if p.is_file():
        print(f'  {p.name}: {p.stat().st_size / 1e6:.1f} MB')

## Step 6: 推理测试

In [ ]:
model.config.use_cache = True
model.eval()

if ROLE_PROMPT_PATH.exists():
    system_prompt = ROLE_PROMPT_PATH.read_text(encoding='utf-8')
else:
    system_prompt = '你是一名临床药师，请专业、循证地回答用药安全问题。'

test_qs = [
    '孕妇感冒能吃布洛芬吗？',
    '高血压患者同时吃阿司匹林和华法林有风险吗？',
    '小孩 3 岁 15kg 发烧 39 度，对乙酰氨基酚怎么吃？',
]

for q in test_qs:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': q},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=512,
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n=== 问: {q} ===\n答:\n{response}\n{"-"*80}')